# INC-005 | Dados de fornecedor divergem entre consultas

## Relato do cliente
"A equipe de compras encontrou nomes e telefones diferentes para o mesmo fornecedor ao comparar o cadastro com a ficha de uma peça."

## Hipótese inicial
O modelo relacional do sistema legado viola a **Terceira Forma Normal (3FN)** ao replicar atributos de fornecedor (`fornecedor_nome_cache` e `fornecedor_telefone_cache`) diretamente na tabela `industria.componentes`, em vez de depender exclusivamente da chave primária através da chave estrangeira `fornecedor_id`. A ausência de um mecanismo de propagação automática (como uma trigger ou rotina transacional sincronizada) causa **anomalias de atualização**: quando o cadastro oficial em `industria.fornecedores` é atualizado, a tabela `componentes` permanece com dados defasados.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=55432,
        dbname="pbl_industria_testes",
        user="aluno",
        password=config.get("POSTGRES_PASSWORD", "pipe101198")
    )

## 1. Mapeamento da estrutura de dados

Ao analisar os metadados das tabelas `industria.componentes` e `industria.fornecedores`, identificamos colunas redundantes:

- **Tabela `fornecedores`**: `id`, `nome`, `cnpj`, `telefone`, `email`, `ativo`.
- **Tabela `componentes`**: `id`, `codigo`, `descricao`, `fornecedor_id`, `fornecedor_nome_cache`, `fornecedor_telefone_cache`, `preco_atual`.

A função `listar_componentes` em `aplicacao/operacoes.py` (linhas 25 a 38) evidencia a coexistência das duas fontes de dados:

```sql
SELECT c.codigo, c.descricao, c.preco_atual,
       f.nome AS fornecedor_cadastrado,
       c.fornecedor_nome_cache AS fornecedor_na_ficha,
       f.telefone AS telefone_cadastrado,
       c.fornecedor_telefone_cache AS telefone_na_ficha
FROM industria.componentes c
JOIN industria.fornecedores f ON f.id = c.fornecedor_id
ORDER BY c.codigo;
```

In [ ]:
# Identificação das divergências existentes na base de testes
sql_divergencias = """
SELECT
    c.codigo AS componente,
    f.id AS fornecedor_id,
    f.nome AS nome_oficial,
    c.fornecedor_nome_cache AS nome_na_ficha,
    f.telefone AS telefone_oficial,
    c.fornecedor_telefone_cache AS telefone_na_ficha
FROM industria.componentes c
JOIN industria.fornecedores f ON f.id = c.fornecedor_id
WHERE f.nome <> c.fornecedor_nome_cache
   OR f.telefone <> c.fornecedor_telefone_cache;
"""

with conectar_testes() as conn:
    df_divergencias = pd.read_sql(sql_divergencias, conn)

print(f"Total de componentes com divergência de fornecedor: {len(df_divergencias)}")
display(df_divergencias)

## 2. Reprodução experimental da anomalia de atualização

Demonstra-se abaixo como a atualização no cadastro oficial de um fornecedor gera inconsistência imediata nas fichas dos componentes associados.

In [ ]:
# 1. Escolher um fornecedor de teste e atualizar seus dados cadastrais
fornecedor_alvo = 1

with conectar_testes() as conn:
    with conn.cursor() as cur:
        # Atualizar nome e telefone do fornecedor
        cur.execute("""
            UPDATE industria.fornecedores
            SET nome = 'Fornecedor Industrial Alfa Atualizado S/A',
                telefone = '(11) 99999-0000'
            WHERE id = %s;
        """, (fornecedor_alvo,))
        conn.commit()

# 2. Consultar a ficha do componente vinculado
with conectar_testes() as conn:
    df_verificacao = pd.read_sql("""
        SELECT
            c.codigo,
            f.nome AS nome_cadastro,
            c.fornecedor_nome_cache AS nome_ficha,
            f.telefone AS tel_cadastro,
            c.fornecedor_telefone_cache AS tel_ficha
        FROM industria.componentes c
        JOIN industria.fornecedores f ON f.id = c.fornecedor_id
        WHERE f.id = %s;
    """, conn, params=[fornecedor_alvo])

display(df_verificacao)
print("Divergência gerada: o cadastro oficial foi alterado, mas o cache na tabela componentes permaneceu estático!")

## 3. Soluções propostas

Apresentamos duas alternativas técnicas para solucionar o problema:

### Abordagem A: Normalização Arquitetural Completa (Recomendada)
Eliminar as colunas `fornecedor_nome_cache` e `fornecedor_telefone_cache` da tabela `industria.componentes`. As consultas que necessitarem dessas informações devem sempre realizar a junção natural com a tabela oficial `fornecedores`:

```sql
ALTER TABLE industria.componentes
DROP COLUMN fornecedor_nome_cache,
DROP COLUMN fornecedor_telefone_cache;
```

### Abordagem B: Sincronização Automática via Trigger (Preservação de Legado)
Caso sistemas externos dependam estritamente da estrutura de colunas atual, cria-se uma trigger no PostgreSQL para sincronizar automaticamente os campos de cache:

```sql
CREATE OR REPLACE FUNCTION industria.fn_sincronizar_fornecedor_componentes()
RETURNS TRIGGER AS $$
BEGIN
    UPDATE industria.componentes
    SET fornecedor_nome_cache = NEW.nome,
        fornecedor_telefone_cache = NEW.telefone
    WHERE fornecedor_id = NEW.id;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_sync_fornecedor_componentes
AFTER UPDATE OF nome, telefone ON industria.fornecedores
FOR EACH ROW
EXECUTE FUNCTION industria.fn_sincronizar_fornecedor_componentes();
```

In [ ]:
# Validação da abordagem com sincronização via trigger
sql_trigger = """
CREATE OR REPLACE FUNCTION industria.fn_sincronizar_fornecedor_componentes()
RETURNS TRIGGER AS $$
BEGIN
    UPDATE industria.componentes
    SET fornecedor_nome_cache = NEW.nome,
        fornecedor_telefone_cache = NEW.telefone
    WHERE fornecedor_id = NEW.id;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

DROP TRIGGER IF EXISTS trg_sync_fornecedor_componentes ON industria.fornecedores;

CREATE TRIGGER trg_sync_fornecedor_componentes
AFTER UPDATE OF nome, telefone ON industria.fornecedores
FOR EACH ROW
EXECUTE FUNCTION industria.fn_sincronizar_fornecedor_componentes();
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_trigger)
        
        # Testar atualização sob efeito da trigger
        cur.execute("""
            UPDATE industria.fornecedores
            SET nome = 'Alfa Automação e Peças Ltda',
                telefone = '(11) 3333-4444'
            WHERE id = 1;
        """)
        conn.commit()

# Verificar se os componentes foram atualizados automaticamente
with conectar_testes() as conn:
    df_pos_trigger = pd.read_sql("""
        SELECT c.codigo, f.nome AS oficial, c.fornecedor_nome_cache AS ficha,
               f.telefone AS tel_oficial, c.fornecedor_telefone_cache AS tel_ficha
        FROM industria.componentes c
        JOIN industria.fornecedores f ON f.id = c.fornecedor_id
        WHERE f.id = 1;
    """, conn)

display(df_pos_trigger)
assert (df_pos_trigger["oficial"] == df_pos_trigger["ficha"]).all(), "Sincronização falhou!"
print("Validação com trigger: Sucesso! Os campos de cache foram sincronizados instantaneamente.")

# Ficha técnica | INC-005

## 1. Hipótese e resultado esperado
A equipe de compras encontrou divergências de nomes e telefones para o mesmo fornecedor entre o cadastro e a ficha da peça.
- **Hipótese:** Violação da 3ª Forma Normal (desnormalização de atributos com colunas de cache na tabela `componentes`) sem garantia de integridade ou propagação de atualizações.
- **Resultado esperado:** As informações cadastrais de um fornecedor devem ser únicas e consistentes em qualquer ponto de consulta do sistema.

## 2. Procedimento de investigação
1. Mapeamento de colunas em `industria.componentes` e `industria.fornecedores`.
2. Execução de consulta SQL filtrando registros onde `fornecedor_nome_cache <> fornecedores.nome` ou telefone correspondente.
3. Simulação de atualização de fornecedor via comando SQL, constatando a permanência dos dados defasados na ficha do componente.

## 3. Evidências e resultados
- Foram localizados múltiplos componentes com discrepância ativa entre o cadastro de fornecedores e as colunas de cache da tabela `componentes`.
- A alteração cadastral no fornecedor comprovou a anomalia de atualização.
- A implementação da trigger `trg_sync_fornecedor_componentes` demonstrou que as alterações passam a ser replicadas em tempo real e de forma atômica no SGBD.

## 4. Conclusão
Diagnóstico confirmado. A inconsistência decorre de redundância não controlada de dados no schema relacional.

## 5. Solução proposta e validação
1. A longo prazo: Normalizar o banco removendo as colunas de cache de `componentes` (3FN).
2. A curto prazo: Implantar a trigger de sincronização automática no PostgreSQL e rodar rotina de carga para sanear os dados existentes.
3. Teste de validação: Atualizar fornecedores e verificar coincidência de 100% dos registros.